In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv('./predictions_full2.csv')
df.head()

,Id,roberta_neg,roberta_neu,roberta_pos,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,ProductURL,review_datetime,cleaned_text,review_word_count,pred_label
0,1,0.009624,0.049980,0.940395,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,https://www.amazon.com/dp/B001E4KFG0,2011-04-27,I have bought several of the Vitality canned d...,48,positive
1,2,0.508986,0.452414,0.038600,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,https://www.amazon.com/dp/B00813GRG4,2012-09-07,Product arrived labeled as Jumbo Salted Peanut...,31,negative
2,3,0.003229,0.098068,0.898704,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,https://www.amazon.com/dp/B000LQOCH0,2008-08-18,This is a confection that has been around a fe...,94,positive
3,4,0.002295,0.090219,0.907485,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,https://www.amazon.com/dp/B000UA0QIQ,2011-06-13,If you are looking for the secret ingredient i...,41,positive
4,5,0.001635,0.010302,0.988063,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,https://www.amazon.com/dp/B006K2ZZ7K,2012-10-21,Great taffy at a great price. There was a wide...,27,positive


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 567228 entries, 0 to 567227
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Id                      567228 non-null  int64  
 1   roberta_neg             567228 non-null  float64
 2   roberta_neu             567228 non-null  float64
 3   roberta_pos             567228 non-null  float64
 4   ProductId               567228 non-null  str    
 5   UserId                  567228 non-null  str    
 6   ProfileName             567180 non-null  str    
 7   HelpfulnessNumerator    567228 non-null  int64  
 8   HelpfulnessDenominator  567228 non-null  int64  
 9   Score                   567228 non-null  int64  
 10  Time                    567228 non-null  int64  
 11  Summary                 567199 non-null  str    
 12  Text                    567228 non-null  str    
 13  ProductURL              567228 non-null  str    
 14  review_datetime         567228 

In [ ]:
result_df = df[['cleaned_text', 'Score', 'pred_label']]
result_df.to_csv('./cleaned_pred_score2.csv', index=False)
result_df.head()

,cleaned_text,Score,pred_label
0,I have bought several of the Vitality canned d...,5,positive
1,Product arrived labeled as Jumbo Salted Peanut...,1,negative
2,This is a confection that has been around a fe...,4,positive
3,If you are looking for the secret ingredient i...,2,positive
4,Great taffy at a great price. There was a wide...,5,positive


In [ ]:
def star_to_label(score):
    if score >= 4:
        return 'positive'
    if score == 3:
        return 'neutral'
    if score <= 2:
        return 'negative'
    return 'neutral'

# Expected label from score
df['expected_label'] = df['Score'].apply(star_to_label)

# Count predicted labels for each score
score_pred_counts = (
    df.groupby(['Score', 'pred_label'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=['positive', 'neutral', 'negative'], fill_value=0)
      .sort_index()
)

# Save counts to CSV
score_pred_counts.to_csv('./score_pred_label_counts2.csv')

# Optional: compare expected-label counts by score
score_expected_counts = (
    df.groupby(['Score', 'expected_label'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=['positive', 'neutral', 'negative'], fill_value=0)
      .sort_index()
)

print('Predicted label counts per score:')
display(score_pred_counts)

print('Expected label counts per score (from star_to_label):')
display(score_expected_counts)

Predicted label counts per score:


pred_label,positive,neutral,negative
Score,,,
1,4825,5149,41996
2,5585,5092,19076
3,19065,9905,13588
4,67696,8150,4704
5,340962,13632,7803


Expected label counts per score (from star_to_label):


expected_label,positive,neutral,negative
Score,,,
1,0,0,51970
2,0,0,29753
3,0,42558,0
4,80550,0,0
5,362397,0,0


In [ ]:
# Store rows in separate CSV files by score only (1-5)
scores = [1, 2, 3, 4, 5]

score_data = {}

for score in scores:
    subset = result_df[result_df['Score'] == score].copy()
    score_data[score] = subset
    subset.to_csv(f'./score_{score}2.csv', index=False)

# Optional: one combined file for scores 1-5
combined_df = result_df[result_df['Score'].isin(scores)].copy()
combined_df.to_csv('./score_1_to_5_all2.csv', index=False)

# Quick summary table: counts by score and predicted label
summary = (
    combined_df.groupby(['Score', 'pred_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['positive', 'neutral', 'negative'], fill_value=0)
    .sort_index()
)
summary

pred_label,positive,neutral,negative
Score,,,
1,4825,5149,41996
2,5585,5092,19076
3,19065,9905,13588
4,67696,8150,4704
5,340962,13632,7803
